# ClearBank – Análise de Transações

Notebook que lê, valida e analisa o arquivo `transacoes.csv`, gera métricas mensais, sinaliza transações suspeitas, exibe um relatório formatado e salva o resultado em `relatorio.json`.

## Como usar
1. Envie o arquivo `transacoes.csv` para o Colab (ícone de pasta → upload).
2. Execute as células **em ordem**, de cima para baixo (ou use *Ambiente de execução → Executar tudo*).
3. A última célula (**Execução Principal**) roda o processo completo.

## Estrutura do arquivo `transacoes.csv`
`id, data (AAAA-MM-DD), cliente_id, tipo (credito/debito), valor (> 0), descricao, categoria`

Linhas inválidas (id vazio/não numérico, cliente vazio, data inválida, tipo inválido, valor inválido) são descartadas automaticamente.

In [4]:
import csv
import json
from datetime import datetime, date

ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"
LIMITE_SUSPEITO = 10000.00
FORMATO_DATA = "%Y-%m-%d"
TIPOS_VALIDOS = ("credito", "debito")

print("Constantes carregadas com sucesso.")

Constantes carregadas com sucesso.


In [5]:
def ler_transacoes(caminho):
    """Lê o CSV e devolve a lista de linhas brutas (cada linha é um dicionário)."""
    try:
        with open(caminho, newline="", encoding="utf-8") as arquivo:
            return list(csv.DictReader(arquivo))
    except FileNotFoundError:
        print(f"Erro: o arquivo '{caminho}' não foi encontrado.")
    except UnicodeDecodeError:
        print(f"Erro: o arquivo '{caminho}' não está codificado em UTF-8.")
    return []


# --- Teste rápido ---
linhas_teste = ler_transacoes(ARQUIVO_CSV)
print("Linhas lidas:", len(linhas_teste))
print("Primeira linha:", linhas_teste[0])
print("Arquivo inexistente:", ler_transacoes("nao_existe.csv"))

Linhas lidas: 25
Primeira linha: {'id': '1', 'data': '2026-01-05', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500.00', 'descricao': 'Salário janeiro', 'categoria': 'salario'}
Erro: o arquivo 'nao_existe.csv' não foi encontrado.
Arquivo inexistente: []


In [6]:
def converter_id(texto):
    """Converte o texto em inteiro. Retorna None se estiver vazio ou não for número."""
    try:
        return int(texto)
    except (ValueError, TypeError):
        return None


def converter_valor(texto):
    """Converte o texto em float. Retorna None se não for número ou se for <= 0."""
    try:
        valor = float(texto)
    except (ValueError, TypeError):
        return None
    return valor if valor > 0 else None


def converter_data(texto):
    """Converte 'AAAA-MM-DD' em datetime. Retorna None se o formato for inválido."""
    try:
        if len(texto) != 10:  # strptime aceitaria '2026-1-5'; aqui exigimos AAAA-MM-DD
            return None
        return datetime.strptime(texto, FORMATO_DATA)
    except (ValueError, TypeError):
        return None


def validar_transacao(linha):
    """Valida uma linha bruta. Retorna o registro limpo (dict) ou None se for inválida."""
    id_transacao = converter_id(linha.get("id"))
    cliente_id = (linha.get("cliente_id") or "").strip()
    tipo = (linha.get("tipo") or "").strip().lower()
    valor = converter_valor(linha.get("valor"))
    data = converter_data(linha.get("data"))

    if id_transacao is None or not cliente_id or tipo not in TIPOS_VALIDOS:
        return None
    if valor is None or data is None:
        return None

    return {"id": id_transacao, "data": data, "cliente_id": cliente_id,
            "tipo": tipo, "valor": valor}


def limpar_transacoes(linhas):
    """Valida todas as linhas. Retorna (lista_de_validas, quantidade_de_invalidas)."""
    validas = []
    for linha in linhas:
        registro = validar_transacao(linha)
        if registro is not None:
            validas.append(registro)
    return validas, len(linhas) - len(validas)


def exibir_resumo_limpeza(total, validas, invalidas):
    """Mostra o resumo da limpeza dos dados."""
    print(f"Total de linhas lidas: {total}")
    print(f"Linhas válidas: {validas}")
    print(f"Linhas inválidas: {invalidas}")


# --- Teste rápido ---
print("converter_valor('abc')        ->", converter_valor("abc"))
print("converter_valor('-50')        ->", converter_valor("-50"))
print("converter_valor('3500.00')    ->", converter_valor("3500.00"))
print("converter_data('18/02/2026')  ->", converter_data("18/02/2026"))
print("converter_data('2026-02-18')  ->", converter_data("2026-02-18"))
print()
linhas_teste = ler_transacoes(ARQUIVO_CSV)
validas_teste, qtd_invalidas_teste = limpar_transacoes(linhas_teste)
exibir_resumo_limpeza(len(linhas_teste), len(validas_teste), qtd_invalidas_teste)

converter_valor('abc')        -> None
converter_valor('-50')        -> None
converter_valor('3500.00')    -> 3500.0
converter_data('18/02/2026')  -> None
converter_data('2026-02-18')  -> 2026-02-18 00:00:00

Total de linhas lidas: 25
Linhas válidas: 20
Linhas inválidas: 5


In [7]:
def extrair_mes(data):
    """Recebe um datetime e devolve o mês no formato AAAA-MM."""
    return data.strftime("%Y-%m")


def calcular_periodo(transacoes):
    """Retorna (data_mais_antiga, data_mais_recente, dias_entre_elas)."""
    datas = [t["data"] for t in transacoes]
    inicio = min(datas)
    fim = max(datas)
    return inicio, fim, (fim - inicio).days


# --- Teste rápido ---
validas_teste, _ = limpar_transacoes(ler_transacoes(ARQUIVO_CSV))
inicio, fim, dias = calcular_periodo(validas_teste)
print("Transação mais antiga:", inicio.strftime(FORMATO_DATA))
print("Transação mais recente:", fim.strftime(FORMATO_DATA))
print("Dias entre elas:", dias)
print("Mês da primeira transação:", extrair_mes(validas_teste[0]["data"]))

Transação mais antiga: 2026-01-05
Transação mais recente: 2026-04-25
Dias entre elas: 110
Mês da primeira transação: 2026-01


In [8]:
def agrupar_por_mes(transacoes):
    """Agrupa as transações por mês. Retorna {'AAAA-MM': [transações]}."""
    grupos = {}
    for transacao in transacoes:
        mes = extrair_mes(transacao["data"])
        grupos.setdefault(mes, []).append(transacao)
    return grupos


def calcular_metricas(transacoes_do_mes):
    """Calcula as métricas financeiras de um mês."""
    valores = [t["valor"] for t in transacoes_do_mes]
    credito = sum(t["valor"] for t in transacoes_do_mes if t["tipo"] == "credito")
    debito = sum(t["valor"] for t in transacoes_do_mes if t["tipo"] == "debito")
    return {
        "quantidade": len(valores),
        "total_credito": round(credito, 2),
        "total_debito": round(debito, 2),
        "saldo": round(credito - debito, 2),
        "media": round(sum(valores) / len(valores), 2),
        "maior_valor": max(valores),
        "menor_valor": min(valores),
    }


def identificar_suspeitas(transacoes):
    """Retorna a lista de transações com valor acima de LIMITE_SUSPEITO."""
    suspeitas = []
    for t in transacoes:
        if t["valor"] > LIMITE_SUSPEITO:
            suspeitas.append({
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime(FORMATO_DATA),
                "valor": t["valor"],
            })
    return suspeitas


def gerar_relatorio(transacoes, qtd_invalidas):
    """Monta o dicionário completo do relatório."""
    inicio, fim, dias = calcular_periodo(transacoes)
    grupos = agrupar_por_mes(transacoes)
    resumo_mensal = {mes: calcular_metricas(grupos[mes]) for mes in sorted(grupos)}
    return {
        "gerado_em": date.today().isoformat(),
        "total_transacoes_validas": len(transacoes),
        "total_transacoes_invalidas": qtd_invalidas,
        "resumo_mensal": resumo_mensal,
        "periodo": {
            "inicio": inicio.strftime(FORMATO_DATA),
            "fim": fim.strftime(FORMATO_DATA),
            "dias": dias,
        },
        "transacoes_suspeitas": identificar_suspeitas(transacoes),
    }


# --- Teste rápido ---
validas_teste, invalidas_teste = limpar_transacoes(ler_transacoes(ARQUIVO_CSV))
relatorio_teste = gerar_relatorio(validas_teste, invalidas_teste)
print(json.dumps(relatorio_teste["resumo_mensal"]["2026-01"], indent=2, ensure_ascii=False))
print("Meses encontrados:", list(relatorio_teste["resumo_mensal"].keys()))
print("Transações suspeitas:", len(relatorio_teste["transacoes_suspeitas"]))

{
  "quantidade": 6,
  "total_credito": 10500.0,
  "total_debito": 1470.4,
  "saldo": 9029.6,
  "media": 1995.07,
  "maior_valor": 4200.0,
  "menor_valor": 89.9
}
Meses encontrados: ['2026-01', '2026-02', '2026-03', '2026-04']
Transações suspeitas: 3


In [9]:
def salvar_json(relatorio, caminho):
    """Salva o relatório em um arquivo JSON."""
    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)
    print(f"Relatório salvo em '{caminho}'.")


# --- Teste rápido ---
salvar_json(relatorio_teste, ARQUIVO_JSON)
with open(ARQUIVO_JSON, encoding="utf-8") as arquivo:
    print(arquivo.read()[:350])

Relatório salvo em 'relatorio.json'.
{
  "gerado_em": "2026-10-06",
  "total_transacoes_validas": 20,
  "total_transacoes_invalidas": 5,
  "resumo_mensal": {
    "2026-01": {
      "quantidade": 6,
      "total_credito": 10500.0,
      "total_debito": 1470.4,
      "saldo": 9029.6,
      "media": 1995.07,
      "maior_valor": 4200.0,
      "menor_valor": 89.9
    },
    "2026-02": {
 


In [10]:
def formatar_moeda(valor):
    """Formata um número no padrão brasileiro: R$ 1.234,56."""
    texto = f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return f"R$ {texto}"


def exibir_periodo(relatorio):
    """Mostra o período analisado e os totais de transações."""
    periodo = relatorio["periodo"]
    print("=" * 50)
    print("RELATÓRIO CLEARBANK - ANÁLISE DE TRANSAÇÕES")
    print("=" * 50)
    print(f"Período analisado: {periodo['inicio']} → {periodo['fim']} ({periodo['dias']} dias)")
    print(f"Transações válidas:   {relatorio['total_transacoes_validas']}")
    print(f"Transações inválidas: {relatorio['total_transacoes_invalidas']}")


def exibir_mes(mes, dados):
    """Mostra as métricas de um mês."""
    print(f"\nMês: {mes}")
    print(f"  Transações: {dados['quantidade']}")
    print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
    print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
    print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
    print(f"  Média:         {formatar_moeda(dados['media'])}")
    print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
    print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}")


def exibir_suspeitas(suspeitas):
    """Mostra a lista de transações suspeitas."""
    print("\n===== TRANSAÇÕES SUSPEITAS =====")
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
        return
    for s in suspeitas:
        print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | "
              f"Data: {s['data']} | Valor: {formatar_moeda(s['valor'])}")


def exibir_relatorio(relatorio):
    """Imprime o relatório completo no terminal."""
    exibir_periodo(relatorio)
    print("\n===== RELATÓRIO MENSAL =====")
    for mes, dados in relatorio["resumo_mensal"].items():
        exibir_mes(mes, dados)
    exibir_suspeitas(relatorio["transacoes_suspeitas"])


# --- Teste rápido ---
print("formatar_moeda(1840.25) ->", formatar_moeda(1840.25))
print()
exibir_relatorio(relatorio_teste)

formatar_moeda(1840.25) -> R$ 1.840,25

RELATÓRIO CLEARBANK - ANÁLISE DE TRANSAÇÕES
Período analisado: 2026-01-05 → 2026-04-25 (110 dias)
Transações válidas:   20
Transações inválidas: 5

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 6
  Total crédito: R$ 10.500,00
  Total débito:  R$ 1.470,40
  Saldo:         R$ 9.029,60
  Média:         R$ 1.995,07
  Maior valor:   R$ 4.200,00
  Menor valor:   R$ 89,90

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 20.100,00
  Total débito:  R$ 1.770,75
  Saldo:         R$ 18.329,25
  Média:         R$ 4.374,15
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 250,75

Mês: 2026-03
  Transações: 5
  Total crédito: R$ 26.000,00
  Total débito:  R$ 10.550,20
  Saldo:         R$ 15.449,80
  Média:         R$ 7.310,04
  Maior valor:   R$ 22.500,00
  Menor valor:   R$ 99,90

Mês: 2026-04
  Transações: 4
  Total crédito: R$ 5.550,00
  Total débito:  R$ 13.580,00
  Saldo:         R$ -8.030,00
  Média:         R$ 4.782,50
  Maior valor:   R$ 12.

In [11]:
def main():
    linhas_brutas = ler_transacoes(ARQUIVO_CSV)
    transacoes_validas, qtd_invalidas = limpar_transacoes(linhas_brutas)
    exibir_resumo_limpeza(len(linhas_brutas), len(transacoes_validas), qtd_invalidas)

    if not transacoes_validas:
        print("Nenhuma transação válida para analisar.")
        return

    relatorio = gerar_relatorio(transacoes_validas, qtd_invalidas)
    salvar_json(relatorio, ARQUIVO_JSON)
    print()
    exibir_relatorio(relatorio)


main()

Total de linhas lidas: 25
Linhas válidas: 20
Linhas inválidas: 5
Relatório salvo em 'relatorio.json'.

RELATÓRIO CLEARBANK - ANÁLISE DE TRANSAÇÕES
Período analisado: 2026-01-05 → 2026-04-25 (110 dias)
Transações válidas:   20
Transações inválidas: 5

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 6
  Total crédito: R$ 10.500,00
  Total débito:  R$ 1.470,40
  Saldo:         R$ 9.029,60
  Média:         R$ 1.995,07
  Maior valor:   R$ 4.200,00
  Menor valor:   R$ 89,90

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 20.100,00
  Total débito:  R$ 1.770,75
  Saldo:         R$ 18.329,25
  Média:         R$ 4.374,15
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 250,75

Mês: 2026-03
  Transações: 5
  Total crédito: R$ 26.000,00
  Total débito:  R$ 10.550,20
  Saldo:         R$ 15.449,80
  Média:         R$ 7.310,04
  Maior valor:   R$ 22.500,00
  Menor valor:   R$ 99,90

Mês: 2026-04
  Transações: 4
  Total crédito: R$ 5.550,00
  Total débito:  R$ 13.580,00
  Saldo:         R$